In [2]:
import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/hulumed-14B/accuracy_comparison.json', 'r') as f:
    data = json.load(f)

correct_answers = []
incorrect_answers = []

for item in tqdm(data):
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    model_answer = item['model_answer'].lower().strip()
    match = item['match'].upper()  # Make sure it's uppercase
    
    is_correct = False
    
    # Check "how many" questions - just use the match field
    if "how many" in question:
        if match == 'MATCH':
            is_correct = True
    
    # Check "is used" / "is not used" questions
    elif "used" in question:
        if "is not used" in original_answer and model_answer == "no":
            is_correct = True
        elif "is not used" not in original_answer and "is used" in original_answer and model_answer == "yes":
            is_correct = True
    
    if is_correct:
        correct_answers.append(item)
    else:
        incorrect_answers.append(item)

print(f"Total: {len(data)}")
print(f"Correct: {len(correct_answers)}")
print(f"Incorrect: {len(incorrect_answers)}")
print(f"Accuracy: {(len(correct_answers)/len(data)*100):.2f}%")


100%|██████████| 7652/7652 [00:00<00:00, 796891.72it/s]

Total: 7652
Correct: 6953
Incorrect: 699
Accuracy: 90.87%


In [5]:
print(len(prompt_data))
print(len(data))

7652
7652


In [ ]:
with open('/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_4B_answerable_answers.json', 'r') as f:
    prompt_data = json.load(f)


import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/hulumed-4B/accuracy_comparison.json', 'r') as f:
    data = json.load(f)

correct_answers = []
incorrect_answers = []

for item in tqdm(data):
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    model_answer = item['model_answer'].lower().strip()
    match = item['match'].upper()  # Make sure it's uppercase
    
    is_correct = False
    
    # Check "how many" questions - just use the match field
    if "how many" in question:
        if match == 'MATCH':
            is_correct = True
    
    # Check "is used" / "is not used" questions
    elif "used" in question:
        if "is not used" in original_answer and model_answer == "no":
            is_correct = True
        elif "is not used" not in original_answer and "is used" in original_answer and model_answer == "yes":
            is_correct = True
    
    if is_correct:
        correct_answers.append(item)
    else:
        incorrect_answers.append(item)

print(f"Total: {len(data)}")
print(f"Correct: {len(correct_answers)}")
print(f"Incorrect: {len(incorrect_answers)}")
print(f"Accuracy: {(len(correct_answers)/len(data)*100):.2f}%")


In [11]:
import json
from tqdm import tqdm

# Load both files
with open('/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_14B_answerable_answers.json', 'r') as f:
    prompt_data = json.load(f)

with open('/home/as5606/projects/Hulu-Med/hulumed-14B/accuracy_comparison.json', 'r') as f:
    data = json.load(f)

# Create mapping of ID to answerable answers for quick lookup
answerable_map = {item['id']: item for item in prompt_data}


# Abstention patterns
abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
]


correct_answers = []
incorrect_answers = []
abstention_downgrades = []  # Correct answers that became incorrect

for item in tqdm(data):
    item_id = item['id']
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    model_answer = item['model_answer'].lower().strip()
    match = item['match'].upper()
    
    is_correct = False
    
    # Check "how many" questions
    if "how many" in question:
        if match == 'MATCH':
            is_correct = True
    
    # Check "is used" / "is not used" questions
    elif "used" in question:
        if "is not used" in original_answer and model_answer == "no":
            is_correct = True
        elif "is not used" not in original_answer and "is used" in original_answer and model_answer == "yes":
            is_correct = True
    
    # If correct, check if model abstained in the answerable version
    if is_correct:
        if item_id in answerable_map:
            answerable_answer = answerable_map[item_id]['hulu_med_answer'].lower()
            
            # Check if abstained
            is_abstained = any(pattern in answerable_answer for pattern in abstention_patterns)
            
            if is_abstained:
                # Downgrade: was correct but model abstained
                is_correct = False
                abstention_downgrades.append({
                    'id': item_id,
                    'question': item['question'],
                    'original_answer': item['original_answer'],
                    'accuracy_answer': item['model_answer'],
                    'answerable_answer': answerable_map[item_id]['hulu_med_answer']
                })
    
    if is_correct:
        correct_answers.append(item)
    else:
        incorrect_answers.append(item)

# Print summary
print(f"{'='*70}")
print(f"COMBINED EVALUATION (Accuracy + Answerable)")
print(f"{'='*70}")
print(f"Total questions: {len(data)}")
print(f"Correct (final): {len(correct_answers)}")
print(f"Incorrect (final): {len(incorrect_answers)}")
print(f"Downgrades (was correct, but abstained): {len(abstention_downgrades)}")
print(f"Final Accuracy: {(len(correct_answers)/len(data)*100):.2f}%")
print(f"{'='*70}\n")

100%|██████████| 7652/7652 [00:00<00:00, 502360.60it/s]

COMBINED EVALUATION (Accuracy + Answerable)
Total questions: 7652
Correct (final): 6867
Incorrect (final): 785
Downgrades (was correct, but abstained): 86
Final Accuracy: 89.74%



In [50]:
import json
from tqdm import tqdm

# ============= VARIABLES - CHANGE THESE =============
model_size = "14B"  # Change this: "4B", "7B", "14B"
# ====================================================

# Load both files
with open(f'/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_{model_size}_answerable_answers.json', 'r') as f:
    prompt_data = json.load(f)

with open(f'/home/as5606/projects/Hulu-Med/hulumed-{model_size}/accuracy_comparison.json', 'r') as f:
    data = json.load(f)

# Create mapping of ID to answerable answers for quick lookup
answerable_map = {item['id']: item for item in prompt_data}

# Abstention patterns
abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
]

correct_before = 0
correct_after = 0
total_is_used_questions = 0
abstention_downgrades = []

for item in tqdm(data):
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    model_answer = item['model_answer'].lower().strip()
    match = item['match'].upper()
    item_id = item['id']
    
    # Filter: Only "is used" / "is not used" questions
    if "used" not in question:
        continue
    
    total_is_used_questions += 1
    
    is_correct_before = False
    
    # Check accuracy BEFORE prompt (from accuracy_comparison.json)
    if "is not used" in original_answer and model_answer == "no":
        is_correct_before = True
    elif "is not used" not in original_answer and "is used" in original_answer and model_answer == "yes":
        is_correct_before = True
    
    if is_correct_before:
        correct_before += 1
    
    # Now check accuracy AFTER prompt (from answerable_answers.json)
    is_correct_after = is_correct_before
    
    if is_correct_before and item_id in answerable_map:
        answerable_answer = answerable_map[item_id]['hulu_med_answer'].lower()
        
        # Check if model abstained in answerable version
        is_abstained = any(pattern in answerable_answer for pattern in abstention_patterns)
        
        if is_abstained:
            # Downgrade: was correct but model abstained
            is_correct_after = False
            abstention_downgrades.append({
                'id': item_id,
                'question': item['question'],
                'original_answer': item['original_answer'],
                'answer_without_prompt': item['model_answer'],
                'answer_with_prompt': answerable_map[item_id]['hulu_med_answer']
            })
    
    if is_correct_after:
        correct_after += 1

# Print summary
accuracy_before = (correct_before / total_is_used_questions * 100) if total_is_used_questions > 0 else 0
accuracy_after = (correct_after / total_is_used_questions * 100) if total_is_used_questions > 0 else 0

print(f"{'='*70}")
print(f"'IS USED' / 'IS NOT USED' QUESTIONS - PROMPT IMPACT ({model_size})")
print(f"{'='*70}\n")

print(f"Total 'is used' questions: {total_is_used_questions}\n")

print(f"BEFORE adding prompt:")
print(f"  Correct: {correct_before}")
print(f"  Accuracy: {accuracy_before:.2f}%\n")

print(f"AFTER adding prompt:")
print(f"  Correct: {correct_after}")
print(f"  Accuracy: {accuracy_after:.2f}%\n")

print(f"Downgrades (correct → abstained): {len(abstention_downgrades)}")
print(f"Impact: {accuracy_before - accuracy_after:.2f}% (drop)")
print(f"{'='*70}\n")

# Show examples of downgrades
if abstention_downgrades:
    print(f"Examples of downgrades (first 5):\n")
    for i, item in enumerate(abstention_downgrades[:5], 1):
        print(f"{i}. Question: {item['question']}")
        print(f"   Original answer: {item['original_answer']}")
        print(f"   Without prompt: {item['answer_without_prompt']}")
        print(f"   With prompt: {item['answer_with_prompt']}")
        print()


100%|██████████| 7652/7652 [00:00<00:00, 474578.79it/s]

'IS USED' / 'IS NOT USED' QUESTIONS - PROMPT IMPACT (14B)

Total 'is used' questions: 6705

BEFORE adding prompt:
  Correct: 6310
  Accuracy: 94.11%

AFTER adding prompt:
  Correct: 6279
  Accuracy: 93.65%

Downgrades (correct → abstained): 31
Impact: 0.46% (drop)

Examples of downgrades (first 5):

1. Question: is scissors used in preparation?
   Original answer: scissors is not used in preparation
   Without prompt: no
   With prompt: I don't know

2. Question: is specimen bag used in preparation?
   Original answer: specimen bag is not used in preparation
   Without prompt: no
   With prompt: I don't know

3. Question: is hook used in preparation?
   Original answer: hook is not used in preparation
   Without prompt: no
   With prompt: I don't know

4. Question: is scissors used in calot triangle dissection?
   Original answer: scissors is not used in calot triangle dissection
   Without prompt: no
   With prompt: I don't know

5. Question: is scissors used in calot triangle dissect

In [51]:
import json
from tqdm import tqdm

# ============= VARIABLES - CHANGE THESE =============
model_size = "14B"  # Change this: "4B", "7B", "14B"
# ====================================================

# Load both files
with open(f'/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_{model_size}_answerable_answers.json', 'r') as f:
    prompt_data = json.load(f)

with open(f'/home/as5606/projects/Hulu-Med/hulumed-{model_size}/accuracy_comparison.json', 'r') as f:
    data = json.load(f)

# Create mapping of ID to answerable answers for quick lookup
answerable_map = {item['id']: item for item in prompt_data}

# Abstention patterns
abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
]

correct_before = 0
correct_after = 0
total_how_many_questions = 0
abstention_downgrades = []

for item in tqdm(data):
    question = item['question'].lower()
    match = item['match'].upper()
    item_id = item['id']
    
    # Filter: Only "how many" questions
    if "how many" not in question:
        continue
    
    total_how_many_questions += 1
    
    is_correct_before = False
    
    # Check accuracy BEFORE prompt (from accuracy_comparison.json)
    if match == 'MATCH':
        is_correct_before = True
    
    if is_correct_before:
        correct_before += 1
    
    # Now check accuracy AFTER prompt (from answerable_answers.json)
    is_correct_after = is_correct_before
    
    if is_correct_before and item_id in answerable_map:
        answerable_answer = answerable_map[item_id]['hulu_med_answer'].lower()
        
        # Check if model abstained in answerable version
        is_abstained = any(pattern in answerable_answer for pattern in abstention_patterns)
        
        if is_abstained:
            # Downgrade: was correct but model abstained
            is_correct_after = False
            abstention_downgrades.append({
                'id': item_id,
                'question': item['question'],
                'original_answer': item['original_answer'],
                'answer_without_prompt': item['model_answer'],
                'answer_with_prompt': answerable_map[item_id]['hulu_med_answer']
            })
    
    if is_correct_after:
        correct_after += 1

# Print summary
accuracy_before = (correct_before / total_how_many_questions * 100) if total_how_many_questions > 0 else 0
accuracy_after = (correct_after / total_how_many_questions * 100) if total_how_many_questions > 0 else 0

print(f"{'='*70}")
print(f"'HOW MANY' QUESTIONS - PROMPT IMPACT ({model_size})")
print(f"{'='*70}\n")

print(f"Total 'how many' questions: {total_how_many_questions}\n")

print(f"BEFORE adding prompt:")
print(f"  Correct: {correct_before}")
print(f"  Accuracy: {accuracy_before:.2f}%\n")

print(f"AFTER adding prompt:")
print(f"  Correct: {correct_after}")
print(f"  Accuracy: {accuracy_after:.2f}%\n")

print(f"Downgrades (correct → abstained): {len(abstention_downgrades)}")
print(f"Impact: {accuracy_before - accuracy_after:.2f}% (drop)")
print(f"{'='*70}\n")

# Show examples of downgrades
if abstention_downgrades:
    print(f"Examples of downgrades (first 5):\n")
    for i, item in enumerate(abstention_downgrades[:5], 1):
        print(f"{i}. Question: {item['question']}")
        print(f"   Original answer: {item['original_answer']}")
        print(f"   Without prompt: {item['answer_without_prompt']}")
        print(f"   With prompt: {item['answer_with_prompt']}")
        print()


100%|██████████| 7652/7652 [00:00<00:00, 1245095.02it/s]

'HOW MANY' QUESTIONS - PROMPT IMPACT (14B)

Total 'how many' questions: 947

BEFORE adding prompt:
  Correct: 643
  Accuracy: 67.90%

AFTER adding prompt:
  Correct: 588
  Accuracy: 62.09%

Downgrades (correct → abstained): 55
Impact: 5.81% (drop)

Examples of downgrades (first 5):

1. Question: how many tools are operating?
   Original answer: 0
   Without prompt: zero
   With prompt: I don't know

2. Question: how many tools are operating?
   Original answer: 0
   Without prompt: zero
   With prompt: I don't know

3. Question: how many tools are operating?
   Original answer: 1
   Without prompt: there is one surgical tool present in the scene.
   With prompt: I don't know

4. Question: how many tools are operating?
   Original answer: 0
   Without prompt: zero
   With prompt: I don't know

5. Question: how many tools are operating?
   Original answer: 1
   Without prompt: one
   With prompt: I don't know



In [14]:
import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/hulumed-14B/accuracy_comparison.json', 'r') as f:
    data = json.load(f)

# Initialize counters for each category
categories = {
    'is_used': {'correct': 0, 'total': 0, 'incorrect': []},
    'is_not_used': {'correct': 0, 'total': 0, 'incorrect': []},
    'how_many': {'correct': 0, 'total': 0, 'incorrect': []}
}

for item in tqdm(data):
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    model_answer = item['model_answer'].lower().strip()
    match = item['match'].upper()
    
    is_correct = False
    category = None
    
    # Categorize by question type
    if "how many" in question:
        category = 'how_many'
        if match == 'MATCH':
            is_correct = True
    
    elif "is not used" in original_answer:
        category = 'is_not_used'
        if model_answer == "no":
            is_correct = True
    
    elif "is used" in original_answer:
        category = 'is_used'
        if model_answer == "yes":
            is_correct = True
    
    # Count results
    if category:
        categories[category]['total'] += 1
        if is_correct:
            categories[category]['correct'] += 1
        else:
            categories[category]['incorrect'].append({
                'question': item['question'],
                'original_answer': item['original_answer'],
                'model_answer': item['model_answer']
            })

# Print results
print(f"{'='*70}")
print(f"ACCURACY BY QUESTION TYPE")
print(f"{'='*70}\n")

total_correct = 0
total_questions = 0

for cat_name in ['is_used', 'is_not_used', 'how_many']:
    cat = categories[cat_name]
    accuracy = (cat['correct'] / cat['total'] * 100) if cat['total'] > 0 else 0
    
    print(f"{cat_name.upper().replace('_', ' ')}:")
    print(f"  Total: {cat['total']}")
    print(f"  Correct: {cat['correct']}")
    print(f"  Incorrect: {cat['total'] - cat['correct']}")
    print(f"  Accuracy: {accuracy:.2f}%")
    print()
    
    total_correct += cat['correct']
    total_questions += cat['total']

# Overall accuracy
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
print(f"{'='*70}")
print(f"OVERALL ACCURACY: {overall_accuracy:.2f}%")
print(f"{'='*70}")


100%|██████████| 7652/7652 [00:00<00:00, 783297.07it/s]

ACCURACY BY QUESTION TYPE

IS USED:
  Total: 1402
  Correct: 1078
  Incorrect: 324
  Accuracy: 76.89%

IS NOT USED:
  Total: 5303
  Correct: 5232
  Incorrect: 71
  Accuracy: 98.66%

HOW MANY:
  Total: 947
  Correct: 643
  Incorrect: 304
  Accuracy: 67.90%

OVERALL ACCURACY: 90.87%


In [ ]:
/home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_4B_grounded_accuracy_comparison.json

/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_4B_hallucinatory_analysis.py

/home/as5606/Datasets/Cholec_text_semantic_similar_questions/hulu_med_4B_semantic_variations_accuracy.json

/home/as5606/projects/Hulu-Med/hulumed-4B/accuracy_comparison.json

In [45]:
import json
from tqdm import tqdm

# ============= VARIABLES - CHANGE THESE =============
tool_name = "hook"  # Change this: "scissors", "grasper", etc.
model_size = "14B"           # Change this: "4B", "7B", "14B"
# ====================================================

print("="*80)
print(f"ACCURACY FOR '{tool_name.upper()}' - {model_size}")
print("="*80 + "\n")

# ============================================================================
# 1. GROUNDED QUESTIONS
# ============================================================================
print("1. GROUNDED QUESTIONS")
print("-"*80)

try:
    with open(f'/home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_{model_size}_grounded_accuracy_comparison.json', 'r') as f:
        grounded_data = json.load(f)

    grounded_matches = 0
    grounded_total = 0
    grounded_items = []

    for item in tqdm(grounded_data, desc="Processing grounded"):
        if tool_name in item['original_question'].lower():
            grounded_items.append(item['id'])
            for qa in item.get('grounded_qa_with_matches', []):
                grounded_total += 1
                if qa['match'] == 'MATCH':
                    grounded_matches += 1

    grounded_accuracy = (grounded_matches / grounded_total * 100) if grounded_total > 0 else 0
    print(f"Items with '{tool_name}': {len(grounded_items)}")
    print(f"Total subquestions: {grounded_total}")
    print(f"Correct (MATCH): {grounded_matches}")
    print(f"Accuracy: {grounded_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found for model {model_size}\n")
    grounded_accuracy = grounded_matches = grounded_total = 0

# ============================================================================
# 2. HALLUCINATORY QUESTIONS
# ============================================================================
print("2. HALLUCINATORY QUESTIONS")
print("-"*80)

try:
    with open(f'/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_{model_size}_hallucinatory_accuracy.json', 'r') as f:
        halluc_data = json.load(f)

    halluc_matches = 0
    halluc_total = 0
    halluc_items = []

    for item in tqdm(halluc_data, desc="Processing hallucinatory"):
        if tool_name in item['original_question'].lower():
            halluc_items.append(item['id'])
            for qa in item.get('new_questions_with_matches', []):
                halluc_total += 1
                if qa['match'] == 'MATCH':
                    halluc_matches += 1

    halluc_accuracy = (halluc_matches / halluc_total * 100) if halluc_total > 0 else 0
    print(f"Items with '{tool_name}': {len(halluc_items)}")
    print(f"Total subquestions: {halluc_total}")
    print(f"Correct (MATCH): {halluc_matches}")
    print(f"Accuracy: {halluc_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found for model {model_size}\n")
    halluc_accuracy = halluc_matches = halluc_total = 0

# ============================================================================
# 3. SEMANTIC VARIATIONS
# ============================================================================
print("3. SEMANTIC VARIATIONS")
print("-"*80)

try:
    with open(f'/home/as5606/Datasets/Cholec_text_semantic_similar_questions/hulu_med_{model_size}_semantic_variations_accuracy.json', 'r') as f:
        semantic_data = json.load(f)

    semantic_matches = 0
    semantic_total = 0
    semantic_items = []

    for item in tqdm(semantic_data, desc="Processing semantic"):
        if tool_name in item['original_question'].lower():
            semantic_items.append(item['id'])
            for qa in item.get('semantic_variations_with_matches', []):
                semantic_total += 1
                if qa['match'] == 'MATCH':
                    semantic_matches += 1

    semantic_accuracy = (semantic_matches / semantic_total * 100) if semantic_total > 0 else 0
    print(f"Items with '{tool_name}': {len(semantic_items)}")
    print(f"Total subquestions: {semantic_total}")
    print(f"Correct (MATCH): {semantic_matches}")
    print(f"Accuracy: {semantic_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found for model {model_size}\n")
    semantic_accuracy = semantic_matches = semantic_total = 0

# ============================================================================
# 4. TEST ACCURACY (Original Questions)
# ============================================================================
print("4. TEST ACCURACY (Original Questions)")
print("-"*80)

try:
    with open(f'/home/as5606/projects/Hulu-Med/hulumed-{model_size}/accuracy_comparison.json', 'r') as f:
        test_data = json.load(f)

    test_matches = 0
    test_total = 0
    test_items = []

    for item in tqdm(test_data, desc="Processing test"):
        if tool_name in item['question'].lower():
            test_items.append(item['id'])
            test_total += 1
            
            original_answer = item['original_answer'].lower()
            model_answer = item['model_answer'].lower().strip()
            
            is_correct = False
            
            if "is not used" in original_answer and model_answer == "no":
                is_correct = True
            elif "is not used" not in original_answer and "is used" in original_answer and model_answer == "yes":
                is_correct = True
            
            if is_correct:
                test_matches += 1

    test_accuracy = (test_matches / test_total * 100) if test_total > 0 else 0
    print(f"Questions with '{tool_name}': {len(test_items)}")
    print(f"Total questions: {test_total}")
    print(f"Correct: {test_matches}")
    print(f"Accuracy: {test_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found for model {model_size}\n")
    test_accuracy = test_matches = test_total = 0

# ============================================================================
# SUMMARY
# ============================================================================
print("="*80)
print(f"SUMMARY - {tool_name.upper()} ACCURACY ({model_size})")
print("="*80)
print(f"Grounded Questions:      {grounded_accuracy:.2f}% ({grounded_matches}/{grounded_total})")
print(f"Hallucinatory Questions: {halluc_accuracy:.2f}% ({halluc_matches}/{halluc_total})")
print(f"Semantic Variations:     {semantic_accuracy:.2f}% ({semantic_matches}/{semantic_total})")
print(f"Test Accuracy:           {test_accuracy:.2f}% ({test_matches}/{test_total})")
print("="*80)

all_matches = grounded_matches + halluc_matches + semantic_matches + test_matches
all_total = grounded_total + halluc_total + semantic_total + test_total
overall_avg = (all_matches / all_total * 100) if all_total > 0 else 0
print(f"\nOVERALL AVERAGE: {overall_avg:.2f}% ({all_matches}/{all_total})")



import statistics

# Calculate mean and standard deviation
accuracies = [grounded_accuracy, halluc_accuracy, semantic_accuracy, test_accuracy]
mean_accuracy = statistics.mean(accuracies)
std_dev = statistics.stdev(accuracies) if len(accuracies) > 1 else 0

print(f"\nMean Accuracy: {mean_accuracy:.2f}%")
print(f"Standard Deviation: {std_dev:.2f}%")



ACCURACY FOR 'HOOK' - 14B

1. GROUNDED QUESTIONS
--------------------------------------------------------------------------------


Processing grounded: 100%|██████████| 1402/1402 [00:00<00:00, 1081952.94it/s]


Items with 'hook': 518
Total subquestions: 2066
Correct (MATCH): 1196
Accuracy: 57.89%

2. HALLUCINATORY QUESTIONS
--------------------------------------------------------------------------------


Processing hallucinatory: 100%|██████████| 7652/7652 [00:00<00:00, 1896968.75it/s]


Items with 'hook': 934
Total subquestions: 1868
Correct (MATCH): 778
Accuracy: 41.65%

3. SEMANTIC VARIATIONS
--------------------------------------------------------------------------------


Processing semantic: 100%|██████████| 7652/7652 [00:00<00:00, 1819640.22it/s]


Items with 'hook': 934
Total subquestions: 4670
Correct (MATCH): 1124
Accuracy: 24.07%

4. TEST ACCURACY (Original Questions)
--------------------------------------------------------------------------------


Processing test: 100%|██████████| 7652/7652 [00:00<00:00, 1635904.69it/s]

Questions with 'hook': 934
Total questions: 934
Correct: 845
Accuracy: 90.47%

SUMMARY - HOOK ACCURACY (14B)
Grounded Questions:      57.89% (1196/2066)
Hallucinatory Questions: 41.65% (778/1868)
Semantic Variations:     24.07% (1124/4670)
Test Accuracy:           90.47% (845/934)

OVERALL AVERAGE: 41.34% (3943/9538)

Mean Accuracy: 53.52%
Standard Deviation: 28.24%


In [ ]:
#!/usr/bin/env python3
"""
ChatGPT accuracy analysis by equipment/tool.
Analyzes performance across test, grounded, hallucinatory, and semantic variation questions.
"""

import json
from tqdm import tqdm
import statistics

# ============= VARIABLES - CHANGE THESE =============
tool_name = "hook"  # Change this: "scissors", "grasper", "hook", etc.
# ====================================================

print("="*80)
print(f"CHATGPT ACCURACY FOR '{tool_name.upper()}'")
print("="*80 + "\n")

# ============================================================================
# 1. GROUNDED QUESTIONS
# ============================================================================
print("1. GROUNDED QUESTIONS")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_grounded_accuracy.json', 'r') as f:
        grounded_data = json.load(f)

    grounded_matches = 0
    grounded_total = 0
    grounded_items = []

    for item in tqdm(grounded_data, desc="Processing grounded"):
        if tool_name in item['original_question'].lower():
            grounded_items.append(item['id'])
            for qa in item.get('grounded_qa_with_matches', []):
                grounded_total += 1
                if qa['match'] == 'MATCH':
                    grounded_matches += 1

    grounded_accuracy = (grounded_matches / grounded_total * 100) if grounded_total > 0 else 0
    print(f"Items with '{tool_name}': {len(grounded_items)}")
    print(f"Total subquestions: {grounded_total}")
    print(f"Correct (MATCH): {grounded_matches}")
    print(f"Accuracy: {grounded_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    grounded_accuracy = grounded_matches = grounded_total = 0

# ============================================================================
# 2. HALLUCINATORY QUESTIONS
# ============================================================================
print("2. HALLUCINATORY QUESTIONS")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_hallucinatory_answers.json', 'r') as f:
        halluc_data = json.load(f)

    halluc_matches = 0
    halluc_total = 0
    halluc_items = []
    
    abstention_patterns = [
        "i don't know",
        "i do not know",
        "don't know",
        "cannot answer",
        "cannot determine",
        "unable to",
    ]

    for item in tqdm(halluc_data, desc="Processing hallucinatory"):
        if tool_name in item['original_question'].lower():
            halluc_items.append(item['id'])
            for qa in item.get('new_questions_with_chatgpt_answers', []):
                chatgpt_answer = qa.get('chatgpt_answer', '').lower()
                if chatgpt_answer:
                    halluc_total += 1
                    is_abstained = any(pattern in chatgpt_answer for pattern in abstention_patterns)
                    if is_abstained:
                        halluc_matches += 1

    halluc_accuracy = (halluc_matches / halluc_total * 100) if halluc_total > 0 else 0
    print(f"Items with '{tool_name}': {len(halluc_items)}")
    print(f"Total subquestions: {halluc_total}")
    print(f"Correct (abstained): {halluc_matches}")
    print(f"Accuracy: {halluc_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    halluc_accuracy = halluc_matches = halluc_total = 0

# ============================================================================
# 3. SEMANTIC VARIATIONS
# ============================================================================
print("3. SEMANTIC VARIATIONS")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json', 'r') as f:
        semantic_data = json.load(f)

    semantic_matches = 0
    semantic_total = 0
    semantic_items = []

    for item in tqdm(semantic_data, desc="Processing semantic"):
        if tool_name in item['original_question'].lower():
            semantic_items.append(item['id'])
            original_answer = item['original_answer'].lower()
            
            for qa in item.get('semantic_variations_with_chatgpt_answers', []):
                question = qa.get('question', '').lower()
                chatgpt_answer = qa.get('chatgpt_answer', '').lower()
                match = qa.get('match', '').upper()
                
                semantic_total += 1
                is_correct = False
                
                # Check for "how many" questions - use match field
                if "how many" in question:
                    if match == 'MATCH':
                        is_correct = True
                
                # Check for "is used" / "is not used" questions
                elif "is not used" in original_answer:
                    if "no" in chatgpt_answer:
                        is_correct = True
                elif "is used" in original_answer:
                    if "yes" in chatgpt_answer:
                        is_correct = True
                
                if is_correct:
                    semantic_matches += 1

    semantic_accuracy = (semantic_matches / semantic_total * 100) if semantic_total > 0 else 0
    print(f"Items with '{tool_name}': {len(semantic_items)}")
    print(f"Total subquestions: {semantic_total}")
    print(f"Correct: {semantic_matches}")
    print(f"Accuracy: {semantic_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    semantic_accuracy = semantic_matches = semantic_total = 0

# ============================================================================
# 4. TEST ACCURACY (Original Questions)
# ============================================================================
print("4. TEST ACCURACY (Original Questions)")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json', 'r') as f:
        test_data = json.load(f)

    test_matches = 0
    test_total = 0
    test_items = []

    for item in tqdm(test_data, desc="Processing test"):
        if tool_name in item['question'].lower():
            test_items.append(item['id'])
            
            question = item['question'].lower()
            original_answer = item['original_answer'].lower()
            chatgpt_answer = item['chatgpt_answer'].lower().strip()
            match = item.get('match', '').upper()
            
            test_total += 1
            is_correct = False
            
            # Check for "how many" questions - use match field
            if "how many" in question:
                if match == 'MATCH':
                    is_correct = True
            
            # Check for "is used" / "is not used" questions
            elif "is not used" in original_answer:
                if "no" in chatgpt_answer:
                    is_correct = True
            elif "is used" in original_answer:
                if "yes" in chatgpt_answer:
                    is_correct = True
            
            if is_correct:
                test_matches += 1

    test_accuracy = (test_matches / test_total * 100) if test_total > 0 else 0
    print(f"Questions with '{tool_name}': {len(test_items)}")
    print(f"Total questions: {test_total}")
    print(f"Correct: {test_matches}")
    print(f"Accuracy: {test_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    test_accuracy = test_matches = test_total = 0

# ============================================================================
# SUMMARY
# ============================================================================
print("="*80)
print(f"SUMMARY - {tool_name.upper()} ACCURACY")
print("="*80)
print(f"Grounded Questions:      {grounded_accuracy:.2f}% ({grounded_matches}/{grounded_total})")
print(f"Hallucinatory Questions: {halluc_accuracy:.2f}% ({halluc_matches}/{halluc_total})")
print(f"Semantic Variations:     {semantic_accuracy:.2f}% ({semantic_matches}/{semantic_total})")
print(f"Test Accuracy:           {test_accuracy:.2f}% ({test_matches}/{test_total})")
print("="*80)

all_matches = grounded_matches + halluc_matches + semantic_matches + test_matches
all_total = grounded_total + halluc_total + semantic_total + test_total
overall_avg = (all_matches / all_total * 100) if all_total > 0 else 0
print(f"\nOVERALL AVERAGE: {overall_avg:.2f}% ({all_matches}/{all_total})")

# Calculate mean and standard deviation
accuracies = [grounded_accuracy, halluc_accuracy, semantic_accuracy, test_accuracy]
mean_accuracy = statistics.mean(accuracies)
std_dev = statistics.stdev(accuracies) if len(accuracies) > 1 else 0

print(f"\nMean Accuracy: {mean_accuracy:.2f}%")
print(f"Standard Deviation: {std_dev:.2f}%")
print("="*80 + "\n")
